# ZuCo text-only sentiment baseline

Frozen and fine-tuned baselines for ZuCo Task 1 sentiment with three heads:
**mean** pooling, **cls** token, and the **lstm** text component from
Hollenstein et al. (2021), all on a held-out test set via nested
cross-validation.

Run top to bottom on a **GPU runtime** (*Runtime -> Change runtime type -> GPU*).

## 1. Get the code

Clones on first run, pulls the latest commit afterwards, so re-running this
cell brings in any updates pushed to GitHub.

In [ ]:
REPO = "https://github.com/parmisbathaeiyan/zuco-text-baseline.git"
NAME = "zuco-text-baseline"

import os
if not os.path.exists(NAME):
    !git clone -q $REPO
else:
    !git -C $NAME pull -q
%cd $NAME

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Output folder

Everything goes under one `OUTPUT_DIR`. Each setup writes to its own
subfolder `<head>_<mode>/` (e.g. `mean_frozen/`, `cls_finetune/`,
`lstm_frozen/`), one JSON per backbone. The sweep **skips any result that
already exists**, so finished runs are never recomputed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Parmis/Thesis/Results/zuco_text_baseline_again'
folder_name = 'zuco_results_v2'
OUTPUT_DIR = f'{BASE_DIR}/{folder_name}'
print('output dir:', OUTPUT_DIR)

## 3a. (Optional) reuse existing runs

Already have frozen / fine-tuned results from before? Point `EXISTING` at
those folders and this copies each summary into the matching
`<head>_<mode>/` subfolder (named by model) so the sweep skips them.
Leave it empty to skip.

In [ ]:
import glob, json, shutil

# map 'head_mode' -> a folder of existing *.json summaries
EXISTING = {
    # 'mean_frozen':   '/content/drive/MyDrive/.../old_frozen',
    # 'mean_finetune': '/content/drive/MyDrive/.../old_finetune',
}
for setup, src in EXISTING.items():
    dst = f'{OUTPUT_DIR}/{setup}'; os.makedirs(dst, exist_ok=True)
    for p in glob.glob(f'{src}/*.json'):
        name = json.load(open(p))['model_name'].replace('/', '-') + '.json'
        shutil.copy(p, f'{dst}/{name}'); print('reused', setup, name)

## 4. Run the sweep

Trains every missing (backbone x head x mode). Defaults cover all five
backbones, the three heads and both modes. To run a slice, pass e.g.
`--head lstm`, `--mode frozen`, or `--model-name bert-base-uncased`.

Note: this is the full grid; fine-tuned lstm is the slow part. Trim the lists
if you want results sooner, then re-run to fill the rest later.

In [ ]:
!python run.py --output-dir "$OUTPUT_DIR"

## 4a. (Optional) rerun fine-tuned setups at more epochs

`--overwrite` replaces the existing fine-tune results; `--mode finetune` leaves
every frozen result untouched. `--epochs 10` overrides the default for this run
only. The plots in section 5 then use these results automatically.

In [ ]:
!python run.py --output-dir "$OUTPUT_DIR" --mode finetune --epochs 10 --overwrite

## 5. Comparison plots

Built from the saved summaries, so they refresh instantly as more runs land:

- **overview** heatmap of macro-F1 / accuracy over every backbone x setup
- **scores_by_head** — per mode, heads compared (mean vs cls vs lstm)
- **scores_by_mode** — per head, frozen vs fine-tuned
- **confusion / curves** — per setup, confusion matrices and test curves

In [ ]:
!python plot_results.py --results-dir "$OUTPUT_DIR"

In [ ]:
import glob
from IPython.display import Image, display, Markdown
p = f'{OUTPUT_DIR}/plots'

def show(title, pattern):
    paths = sorted(glob.glob(f'{p}/{pattern}'))
    if paths:
        display(Markdown(f'### {title}'))
        for path in paths:
            display(Image(path))

show('Overview', 'overview_*.png')
show('Heads compared (per mode)', 'scores_by_head_*.png')
show('Frozen vs fine-tuned (per head)', 'scores_by_mode_*.png')
show('Confusion matrices (per setup)', 'confusion_*.png')
show('Test curves (per setup)', 'curves_*.png')

## 6. Score table

Averaged test scores for every saved run.

In [ ]:
import glob, json
rows = []
for path in glob.glob(f'{OUTPUT_DIR}/**/*.json', recursive=True):
    if '/plots/' in path: continue
    s = json.load(open(path))
    rows.append((s.get('head', '?'), s['mode'], s['model_name'],
                 s['accuracy_mean'], s['accuracy_std'], s['macro_f1_mean'], s['macro_f1_std']))
for head, mode, model, am, asd, fm, fsd in sorted(rows):
    print(f'{head:<5} {mode:<9} {model:<32} acc {am:.3f}+/-{asd:.3f}  f1 {fm:.3f}+/-{fsd:.3f}')